# AuditLens: Financial Transaction Anomaly Detection & Forensic Audit Analytics
### End-to-End Forensic Audit Pipeline: SQL Ingestion, Statistical Outlier Analysis, Benford's Law & Machine Learning (Isolation Forest)

---
**Project Objectives:**
1. Ingest cleaned enterprise transactions from the analytical SQL layer (`flagged_transactions`).
2. Perform Exploratory Data Analysis (EDA) on corporate spend patterns, departmental variations, and velocity.
3. Apply **Benford's Law First-Digit Analysis** to identify digital manipulation and anomalous invoicing distributions.
4. Detect statistical spend outliers using **Interquartile Range (IQR)** and **Robust Modified Z-Score (MAD)**.
5. Train an unsupervised **Isolation Forest Machine Learning model** on engineered audit features.
6. Cross-validate multi-layer detection models (SQL Rules vs. Statistical vs. ML Isolation Forest).
7. Generate human-readable audit narratives and export enriched data for Power BI dashboards.

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Set high-resolution plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

DB_PATH = '../data/auditlens.db' if os.path.exists('../data/auditlens.db') else 'data/auditlens.db'
print(f"Database connection configured to: {DB_PATH}")

## 1. Database Connection & Data Ingestion
We pull the 100k+ records from SQLite database table `flagged_transactions` created by our analytical SQL layer.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM flagged_transactions", conn)
conn.close()

df['transaction_date'] = pd.to_datetime(df['transaction_date'])
print(f"Successfully loaded {len(df):,} transactions.")
df.head()

## 2. Feature Engineering for Forensic Analytics
We engineer specific features designed to capture forensic audit risk patterns:
- **Log-transformed Spend:** Captures exponential spend variations without skewing models.
- **Proximity to Approval Limits:** Detects structuring behavior near $5k, $10k, $50k thresholds.
- **First Digit Extractor:** Used for Benford's Law compliance.
- **Clean Round Sum Indicator:** Flags suspicious exact retainer/consulting amounts.

In [ ]:
df['log_amount'] = np.log1p(df['amount'])

# Proximity to $5k, $10k, $50k limits
df['is_structuring_candidate'] = (
    ((df['amount'] >= 4800) & (df['amount'] < 5000)) |
    ((df['amount'] >= 9500) & (df['amount'] < 10000)) |
    ((df['amount'] >= 48000) & (df['amount'] < 50000))
).astype(int)

# Benford First Digit
df['first_digit'] = df['amount'].astype(str).str.extract(r"^([1-9])")[0].astype(float).fillna(0).astype(int)

# Clean Round Sum
df['is_clean_round_sum'] = ((df['amount'] >= 1000) & (df['amount'] % 1000 == 0)).astype(int)

print(f"Structuring candidates: {df['is_structuring_candidate'].sum():,}")
print(f"Clean round sums: {df['is_clean_round_sum'].sum():,}")

## 3. Benford's Law First-Digit Analysis
Benford's Law states that in naturally occurring numerical datasets, the number 1 will be the leading first digit approximately 30.1% of the time, while 9 will appear as the leading first digit only 4.6% of the time:
$$P(d) = \log_{10}\left(1 + \frac{1}{d}\right)$$
Significant deviations from Benford's curve in financial ledgers often indicate manual intervention, fabricated invoices, or policy circumvention.

In [ ]:
digits = list(range(1, 10))
benford_expected = [np.log10(1 + 1 / d) * 100.0 for d in digits]
valid_digits = df[df['first_digit'].isin(digits)]['first_digit']
actual_counts = valid_digits.value_counts(normalize=True).reindex(digits, fill_value=0.0) * 100.0

benford_df = pd.DataFrame({
    'Digit': digits,
    'Benford_Expected_%': np.round(benford_expected, 2),
    'Actual_Observed_%': np.round(actual_counts.values, 2),
    'Variance_%': np.round(actual_counts.values - benford_expected, 2)
})

display(benford_df)

# Visualize Benford's Law Test
fig, ax = plt.subplots(figsize=(9, 4.5))
bar_width = 0.35
x = np.arange(len(digits))
ax.bar(x - bar_width/2, benford_df['Benford_Expected_%'], bar_width, label="Benford's Law (Theoretical)", color="#34495e", alpha=0.85)
ax.bar(x + bar_width/2, benford_df['Actual_Observed_%'], bar_width, label="AuditLens Transactions (Observed)", color="#e74c3c", alpha=0.85)
ax.set_xlabel("Leading Digit (1 - 9)", fontweight="bold")
ax.set_ylabel("Frequency Percentage (%)", fontweight="bold")
ax.set_title("Benford's Law First-Digit Forensic Audit Test", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(digits)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

## 4. Statistical Outlier Detection: IQR & Robust Z-Score
We calculate statistical thresholds conditioned on departmental spend dynamics:
1. **Interquartile Range (IQR):** Identifies values exceeding $Q_3 + 2.5 \times \text{IQR}$.
2. **Modified Z-Score:** Uses Median Absolute Deviation (MAD) for robust outlier detection that resists distortion from extreme values.

In [ ]:
df['stat_iqr_flag'] = 0
df['stat_zscore_flag'] = 0

for dept, grp in df.groupby('department'):
    # Method 1: IQR
    q25 = grp['amount'].quantile(0.25)
    q75 = grp['amount'].quantile(0.75)
    iqr = q75 - q25
    iqr_thresh = q75 + (2.5 * iqr)
    
    # Method 2: MAD Robust Z-Score
    med = grp['amount'].median()
    mad = (grp['amount'] - med).abs().median()
    if mad == 0:
        mad = grp['amount'].std()
    mod_z = 0.6745 * (grp['amount'] - med) / (mad + 1e-6)
    
    mask = (df['department'] == dept)
    df.loc[mask & (df['amount'] > iqr_thresh), 'stat_iqr_flag'] = 1
    df.loc[mask & (mod_z > 4.5), 'stat_zscore_flag'] = 1

print(f"IQR Method Flagged      : {df['stat_iqr_flag'].sum():,} records ({(df['stat_iqr_flag'].sum()/len(df))*100:.2f}%)")
print(f"Robust Z-Score Flagged  : {df['stat_zscore_flag'].sum():,} records ({(df['stat_zscore_flag'].sum()/len(df))*100:.2f}%)")

## 5. Machine Learning: Unsupervised Isolation Forest
We train an **Isolation Forest** ensemble algorithm. Isolation Forest isolates anomalies by randomly selecting a feature and randomly splitting value boundaries. Anomalies require fewer splits to isolate than normal data points.

In [ ]:
features = ['log_amount', 'dept_spend_multiplier', 'is_weekend', 'is_clean_round_sum', 'is_structuring_candidate', 'day_of_week']
X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(
    n_estimators=150,
    contamination=0.045,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

df['ml_iso_forest_flag'] = (iso.fit_predict(X_scaled) == -1).astype(int)
raw_scores = iso.decision_function(X_scaled)
norm_scores = 100.0 * (1.0 - ((raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())))
df['ml_anomaly_score'] = np.round(norm_scores, 1)

print(f"Isolation Forest Flagged: {df['ml_iso_forest_flag'].sum():,} transactions ({(df['ml_iso_forest_flag'].sum()/len(df))*100:.2f}%)")

## 6. Multi-Layer Detection Comparison & Correlation
We cross-analyze how our detection layers (SQL Rules, IQR, Z-Score, Isolation Forest) perform against each other and against ground truth anomaly injects.

In [ ]:
comparison_df = pd.DataFrame({
    'SQL_Rules_Flag': (df['risk_tier'] != 'Low').astype(int),
    'IQR_Statistical': df['stat_iqr_flag'],
    'ZScore_Statistical': df['stat_zscore_flag'],
    'Isolation_Forest_ML': df['ml_iso_forest_flag'],
    'Ground_Truth_Anom': df['ground_truth_anomaly']
})

plt.figure(figsize=(7, 5))
sns.heatmap(comparison_df.corr(), annot=True, cmap='Blues', fmt='.2f', vmin=0, vmax=1)
plt.title("Correlation Matrix of Forensic Detection Methods", fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

## 7. Top 15 Priority Audit Targets Ledger
Below are the top high-risk transactions requiring immediate internal audit investigation.

In [ ]:
top_audit_targets = df.sort_values(by=['risk_score', 'ml_anomaly_score', 'amount'], ascending=[False, False, False]).head(15)
display(top_audit_targets[['transaction_id', 'transaction_date', 'department', 'vendor_name', 'amount', 'risk_tier', 'risk_score', 'ml_anomaly_score', 'primary_flag_reason']])